# Feature engineering

**Цель ноутбука:** превращение сырых данных в финальную матрицу признаков, готовую для обучения моделей.

Выводы из [EDA](01_eda.ipynb):
1. Пустые значения в `TotalCharges` свойственны новым клиентам, поэтому вместо пустого значения стоит назначить общие выплаты равные 0
2. Выбор основной метрики - recall или F1/ROC-AUC, необходимо при сплите учесть дисбаланс классов
3. Видна высокая корреляция между tunure и TotalCharges_num, MonthlyCharges и TotalCharges_num, это необходимо учесть при выборе линейных моделей
4. Признаки-кандидаты на удаление, особенно для линейных моделей, так как слабо влияют на таргет (Churn): `PhoneService`, `MultipleLines`. `gender`, удаляем для исключения дискриминации по полу, к тому же он так же слабо влияет на таргет
5. Признак `PaymentMethod` стоит переделать в бинарный (`Electronic chek` = 0/1), так как остальные группы данного признака имеют одинаковые шансы на отток
6. От признака `InternetService` зависят многие услуги, поэтому они не должны учитываться при `InternetService` = No


## Импорт библиотек

In [1]:
import numpy as np 
import pandas as pd
import sklearn

## Загрузка данных

In [2]:
raw_df = pd.read_csv('../data/raw/churn.csv')
raw_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Исправление типов и пропусков (пункт 1 выводов)

Переведем TotalCharges к числовому типу, заменив NaN на месте пустых ячеек на 0.  
А также проверим весь датасет на отсутвие NaN

In [3]:
df = raw_df
df['TotalCharges_num'] = pd.to_numeric(raw_df['TotalCharges'], errors='coerce').fillna(0)
df = df.drop('TotalCharges', axis=1)
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
Churn               0
TotalCharges_num    0
dtype: int64

## Обработка иерархии услуг (пункт 6 выводов)

От признака `InternetService` зависят следующие признаки: `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`.

Варианты действий с этими признаками:

- **Вариант А** — оставить как есть, три категории ("Yes"/"No"/"No internet service") и кодировать как обычный категориальный признак. Плюс: не теряется информация. Минус: избыточность — "No internet service" уже полностью выводится из отдельного признака InternetService, модель обучается на дублирующейся информации.

- **Вариант Б** — схлопнуть "No internet service" и "No" для каждого из этих признаков, превратив их в честные бинарные (Yes/No), раз сама принадлежность к интернет-тарифу уже есть в отдельном признаке InternetService. Это устраняет избыточность и упрощает пространство признаков.  

Выбираю `вариант Б`, дабы исключить обучение на дублирующейся информации и сделать входные данные более чистыми.

In [4]:
def features_to_bin(features, df):
    for feature in features:
        df[f'{feature}_bin'] = (df[feature] == 'Yes').astype('uint8')
        df = df.drop(feature, axis=1)
    return df

In [5]:
features = [
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies'
]

df = features_to_bin(features, df)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,Contract,...,PaymentMethod,MonthlyCharges,Churn,TotalCharges_num,OnlineSecurity_bin,OnlineBackup_bin,DeviceProtection_bin,TechSupport_bin,StreamingTV_bin,StreamingMovies_bin
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,Month-to-month,...,Electronic check,29.85,No,29.85,0,1,0,0,0,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,One year,...,Mailed check,56.95,No,1889.50,1,0,1,0,0,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Month-to-month,...,Mailed check,53.85,Yes,108.15,1,1,0,0,0,0
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,One year,...,Bank transfer (automatic),42.30,No,1840.75,1,0,1,1,0,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,Month-to-month,...,Electronic check,70.70,Yes,151.65,0,0,0,0,0,0


## Кодирование бинарных категориальных признаков

Будем кодировать следующие признаки: `Partner`, `Dependents`, `PaperlessBilling`, плюс сам `Churn`.  
Так же удаляем признак `gender` для исключения дискриминации по полу

In [6]:
features = ['Partner', 'Dependents', 'PaperlessBilling', 'Churn']
df = features_to_bin(features, df)
df.head()

,customerID,gender,SeniorCitizen,tenure,PhoneService,MultipleLines,InternetService,Contract,PaymentMethod,MonthlyCharges,...,OnlineSecurity_bin,OnlineBackup_bin,DeviceProtection_bin,TechSupport_bin,StreamingTV_bin,StreamingMovies_bin,Partner_bin,Dependents_bin,PaperlessBilling_bin,Churn_bin
0,7590-VHVEG,Female,0,1,No,No phone service,DSL,Month-to-month,Electronic check,29.85,...,0,1,0,0,0,0,1,0,1,0
1,5575-GNVDE,Male,0,34,Yes,No,DSL,One year,Mailed check,56.95,...,1,0,1,0,0,0,0,0,0,0
2,3668-QPYBK,Male,0,2,Yes,No,DSL,Month-to-month,Mailed check,53.85,...,1,1,0,0,0,0,0,0,1,1
3,7795-CFOCW,Male,0,45,No,No phone service,DSL,One year,Bank transfer (automatic),42.30,...,1,0,1,1,0,0,0,0,0,0
4,9237-HQITU,Female,0,2,Yes,No,Fiber optic,Month-to-month,Electronic check,70.70,...,0,0,0,0,0,0,0,0,1,1


In [7]:
df = df.drop('gender', axis=1)
df.head()

,customerID,SeniorCitizen,tenure,PhoneService,MultipleLines,InternetService,Contract,PaymentMethod,MonthlyCharges,TotalCharges_num,OnlineSecurity_bin,OnlineBackup_bin,DeviceProtection_bin,TechSupport_bin,StreamingTV_bin,StreamingMovies_bin,Partner_bin,Dependents_bin,PaperlessBilling_bin,Churn_bin
0,7590-VHVEG,0,1,No,No phone service,DSL,Month-to-month,Electronic check,29.85,29.85,0,1,0,0,0,0,1,0,1,0
1,5575-GNVDE,0,34,Yes,No,DSL,One year,Mailed check,56.95,1889.50,1,0,1,0,0,0,0,0,0,0
2,3668-QPYBK,0,2,Yes,No,DSL,Month-to-month,Mailed check,53.85,108.15,1,1,0,0,0,0,0,0,1,1
3,7795-CFOCW,0,45,No,No phone service,DSL,One year,Bank transfer (automatic),42.30,1840.75,1,0,1,1,0,0,0,0,0,0
4,9237-HQITU,0,2,Yes,No,Fiber optic,Month-to-month,Electronic check,70.70,151.65,0,0,0,0,0,0,0,0,1,1


## Кодирование многозначных категориальных признаков

Закодируем `Contract`, `InternetService` как one-hot для большей безопасности, не задумываясь о возможном порядке

In [8]:
contract_cols = pd.get_dummies(df['Contract'], prefix='Contract')
df = pd.concat([df, contract_cols.astype('uint8')], axis=1)
internet_service_cols = pd.get_dummies(df['InternetService'], prefix='InternetService')
df = pd.concat([df, internet_service_cols.astype('uint8')], axis=1)
df = df.drop(['Contract', 'InternetService'], axis=1)
df.head()

,customerID,SeniorCitizen,tenure,PhoneService,MultipleLines,PaymentMethod,MonthlyCharges,TotalCharges_num,OnlineSecurity_bin,OnlineBackup_bin,...,Partner_bin,Dependents_bin,PaperlessBilling_bin,Churn_bin,Contract_Month-to-month,Contract_One year,Contract_Two year,InternetService_DSL,InternetService_Fiber optic,InternetService_No
0,7590-VHVEG,0,1,No,No phone service,Electronic check,29.85,29.85,0,1,...,1,0,1,0,1,0,0,1,0,0
1,5575-GNVDE,0,34,Yes,No,Mailed check,56.95,1889.50,1,0,...,0,0,0,0,0,1,0,1,0,0
2,3668-QPYBK,0,2,Yes,No,Mailed check,53.85,108.15,1,1,...,0,0,1,1,1,0,0,1,0,0
3,7795-CFOCW,0,45,No,No phone service,Bank transfer (automatic),42.30,1840.75,1,0,...,0,0,0,0,0,1,0,1,0,0
4,9237-HQITU,0,2,Yes,No,Electronic check,70.70,151.65,0,0,...,0,0,1,1,1,0,0,0,1,0


Для `PaymentMethod` реализуем решение из пункта 5 выводов: бинаризация "Electronic check" против всех остальных методов, так как остальные группы показали в EDA примерно одинаковую вероятность оттока.

In [9]:
df['Electronic_check'] = (df['PaymentMethod'] == 'Electronic check').astype('uint8')
df = df.drop('PaymentMethod', axis=1)
df.head()

,customerID,SeniorCitizen,tenure,PhoneService,MultipleLines,MonthlyCharges,TotalCharges_num,OnlineSecurity_bin,OnlineBackup_bin,DeviceProtection_bin,...,Dependents_bin,PaperlessBilling_bin,Churn_bin,Contract_Month-to-month,Contract_One year,Contract_Two year,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Electronic_check
0,7590-VHVEG,0,1,No,No phone service,29.85,29.85,0,1,0,...,0,1,0,1,0,0,1,0,0,1
1,5575-GNVDE,0,34,Yes,No,56.95,1889.50,1,0,1,...,0,0,0,0,1,0,1,0,0,0
2,3668-QPYBK,0,2,Yes,No,53.85,108.15,1,1,0,...,0,1,1,1,0,0,1,0,0,0
3,7795-CFOCW,0,45,No,No phone service,42.30,1840.75,1,0,1,...,0,0,0,0,1,0,1,0,0,0
4,9237-HQITU,0,2,Yes,No,70.70,151.65,0,0,0,...,0,1,1,1,0,0,0,1,0,1


## Признаки-кандидаты на удаление (пункт 4 выводов)

Для линейных моделей, для снижения шума, удалим признаки `PhoneService`, `MultipleLines` из-за их слабого влияния на `Churn`, для моделей, устойчивых к таким шумам (деревья и их ансамбли), возможно, вернем эти признаки для сравнения метрик с этими  признаками и без них

In [10]:
df = df.drop(['PhoneService', 'MultipleLines'], axis=1)
df.head()

,customerID,SeniorCitizen,tenure,MonthlyCharges,TotalCharges_num,OnlineSecurity_bin,OnlineBackup_bin,DeviceProtection_bin,TechSupport_bin,StreamingTV_bin,...,Dependents_bin,PaperlessBilling_bin,Churn_bin,Contract_Month-to-month,Contract_One year,Contract_Two year,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Electronic_check
0,7590-VHVEG,0,1,29.85,29.85,0,1,0,0,0,...,0,1,0,1,0,0,1,0,0,1
1,5575-GNVDE,0,34,56.95,1889.50,1,0,1,0,0,...,0,0,0,0,1,0,1,0,0,0
2,3668-QPYBK,0,2,53.85,108.15,1,1,0,0,0,...,0,1,1,1,0,0,1,0,0,0
3,7795-CFOCW,0,45,42.30,1840.75,1,0,1,1,0,...,0,0,0,0,1,0,1,0,0,0
4,9237-HQITU,0,2,70.70,151.65,0,0,0,0,0,...,0,1,1,1,0,0,0,1,0,1


## Мультиколлинеарность числовых признаков (пункт 3 выводов)

`tenure`/`MonthlyCharges`/`TotalCharges`
Сильно коррелирующий признак - `TotalCharges` можно удалить или использовать регуляризацию для линейных моделей (деревья и их ансамбли более устойчивы к корреляции).  
Будем использовать регуляризацию в линейных моделях, чтобы заведомо не упрощать модель для линейных моделей и использовать один датасет на всех моделях

## Масштабирование числовых признаков

Линейным моделям необходимо масштабирование числовых признаков (`tenure`, `MonthlyCharges`, `TotalCharges`), а деревья и их ансамбли не нужнаются в этом, поэтому будем мастабировать признаки непосредственно перед обучением линейных моделей, помимо этого масштабировать признаки необходимо после разбиения на test/train, чтобы избежать утечки данных

## Удаление признаков, не учавствующих в обучении и классификации

Из всех признаков осталось удалить только `customerID` так как он уникален для каждого клиента и абсолютно не влияет на `Churn`, к тому же он будет сбивать модели от правильной классификации

In [11]:
df = df.drop('customerID', axis=1)
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges_num,OnlineSecurity_bin,OnlineBackup_bin,DeviceProtection_bin,TechSupport_bin,StreamingTV_bin,StreamingMovies_bin,...,Dependents_bin,PaperlessBilling_bin,Churn_bin,Contract_Month-to-month,Contract_One year,Contract_Two year,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Electronic_check
0,0,1,29.85,29.85,0,1,0,0,0,0,...,0,1,0,1,0,0,1,0,0,1
1,0,34,56.95,1889.50,1,0,1,0,0,0,...,0,0,0,0,1,0,1,0,0,0
2,0,2,53.85,108.15,1,1,0,0,0,0,...,0,1,1,1,0,0,1,0,0,0
3,0,45,42.30,1840.75,1,0,1,1,0,0,...,0,0,0,0,1,0,1,0,0,0
4,0,2,70.70,151.65,0,0,0,0,0,0,...,0,1,1,1,0,0,0,1,0,1


Проверим также, что оставшиеся признаки и таргет являются численного типа

In [12]:
df.dtypes

SeniorCitizen                    int64
tenure                           int64
MonthlyCharges                 float64
TotalCharges_num               float64
OnlineSecurity_bin               uint8
OnlineBackup_bin                 uint8
DeviceProtection_bin             uint8
TechSupport_bin                  uint8
StreamingTV_bin                  uint8
StreamingMovies_bin              uint8
Partner_bin                      uint8
Dependents_bin                   uint8
PaperlessBilling_bin             uint8
Churn_bin                        uint8
Contract_Month-to-month          uint8
Contract_One year                uint8
Contract_Two year                uint8
InternetService_DSL              uint8
InternetService_Fiber optic      uint8
InternetService_No               uint8
Electronic_check                 uint8
dtype: object

## Разбиение на train/test (пункт 2 выводов)

Разбиение на train/test выполним с помощью `train_test_split` из `scikit learn`

In [ ]:
train, test = sklearn.model_selection.train_test_split(
    df, train_size=0.7, stratify=df['Churn_bin'], random_state=42
)


print(f'train size: {len(train)}, ({len(train) / len(df) * 100:.02f}%)')
print(f'test size: {len(test)}, ({len(test) / len(df) * 100:.02f}%)')

train size: 4930, (70.00%)
test size: 2113, (30.00%)


Проверим доли классов `Churn_bin` в каждой выборке

In [17]:
print(f'Доля ушедших клиентов в датасете: {df['Churn_bin'].mean() * 100:.02f}%')
print(f'Доля ушедших клиентов в train: {train['Churn_bin'].mean() * 100:.02f}%')
print(f'Доля ушедших клиентов в test: {test['Churn_bin'].mean() * 100:.02f}%')

Доля ушедших клиентов в датасете: 26.54%
Доля ушедших клиентов в train: 26.53%
Доля ушедших клиентов в test: 26.55%


## Сохранение результата

In [19]:
PATH = '../data/processed'

train.to_csv(f'{PATH}/train.csv', index=False)
test.to_csv(f'{PATH}/test.csv', index=False)

## Итоговые выводы

1. Для `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` схлопнуты "No internet service" и "No" для каждого из этих признаков, превратив их в честные бинарные (Yes/No), раз сама принадлежность к интернет-тарифу уже есть в отдельном признаке `InternetService`. Это устраняет избыточность и упрощает пространство признаков.  
2. `Partner`, `Dependents`, `PaperlessBilling`, `Churn` переведены в бинарные признаки (0/1), удален `gender` для исключения дискриминации по полу.
3. `Contract`, `InternetService` закодированы как one-hot для большей безопасности, не задумываясь о возможном порядке.
4. Для `PaymentMethod` проведена бинаризация "Electronic check" против всех остальных методов, так как остальные группы показали в EDA примерно одинаковую вероятность оттока, новый признак - `Electronic check`.
5. Для линейных моделей, для снижения шума, удалены признаки `PhoneService`, `MultipleLines` из-за их слабого влияния на `Churn`, для моделей, устойчивых к таким шумам (деревья и их ансамбли), возможно, вернем эти признаки для сравнения метрик с этими  признаками и без них.
6. Корреляция `tenure`/`MonthlyCharges`/`TotalCharges`: будем использовать регуляризацию в линейных моделях, чтобы заведомо не упрощать модель для линейных моделей и использовать один датасет на всех моделях.
7. Линейным моделям необходимо масштабирование числовых признаков (`tenure`, `MonthlyCharges`, `TotalCharges`), а деревья и их ансамбли не нужнаются в этом, поэтому будем масштабировать признаки непосредственно перед обучением линейных моделей, помимо этого масштабировать признаки необходимо после разбиения на test/train, чтобы избежать утечки данных
8. Удален `customerID`